## Step 01 — Download TikTok videos

This notebook reads a CSV of TikTok video URLs, filters them to the months you specify, and downloads each video as an mp4 file using [yt-dlp](https://github.com/yt-dlp/yt-dlp).

An archive file tracks which videos have already been downloaded, so re-running the notebook is safe. It will skip any file that was successfully saved in a previous run.

**Output:** mp4 files saved to `output/videofiles-extracted/`, named by TikTok video ID.


In [ ]:
# INSTALL REQUIREMENTS (one time only):
# %pip install -r ../requirements.txt (locally)
# !pip install yt_dlp #(on google colab)

import json
import subprocess
import time
from pathlib import Path
import random

# If using google colab:
# from google.colab import drive
# drive.mount('/content/drive')

import pandas as pd
import yt_dlp

### Configuration

Make sure to seet **`ORIGINAL_VIDEO_LIST_FILE`** (filename of the input CSV inside the `raw/` folder.) before running the rest of the notebook. For example, if you ran notebook `01` to load in someone's TikTok watch history, you need to rename this variable `watch_history.csv`.


In [ ]:
ORIGINAL_VIDEO_LIST_FILE = "sample_video_list.csv"

OUTPUT_VIDEO_FOLDER = "videofiles-extracted"  # subfolder inside output/ where videos and metadata will be saved

# In Google Drive:
# OUTPUT_PATH = f"/content/drive/MyDrive/tiktok-content-reupload-tracker/output/{OUTPUT_VIDEO_FOLDER}"

# Locally:
OUTPUT_PATH = f"../output/{OUTPUT_VIDEO_FOLDER}"

# Each downloaded file is named using the TikTok video ID (e.g. 7123456789.mp4)
OUTPUT_TEMPLATE = f"{OUTPUT_PATH}/%(id)s.%(ext)s"

# yt-dlp maintains this file to record which video IDs have already been downloaded.
# On future runs it skips any ID listed here, preventing duplicate downloads.
ARCHIVE_FILE = f"{OUTPUT_PATH}/downloaded-list.txt"

### Load video URLs

Read the metadata CSV from the `raw/` folder, filter rows to the months specified above, and extract the `link` column as a plain list that yt-dlp can consume.


In [ ]:

videos = pd.read_csv(f"../raw/{ORIGINAL_VIDEO_LIST_FILE}")
videos.head()

# If you want to filter down to a subset of videos to 
# reduce the number of downloads in one run, you can do so here:
videos_to_scrape = videos
video_urls = videos_to_scrape['link'].tolist()

print("We will scrape a batch of ", len(video_urls), " videos")


### Download

This cell downloads videos with yt-dlp and writes one metadata JSON sidecar per video (`*.info.json`).

Key yt-dlp flags used:

- `--download-archive` - skips any video ID already in `downloaded-list.txt`
- `--add-header Referer` - required by TikTok's CDN to serve the video file
- `--sleep-requests` - adds a delay between requests to avoid rate-limiting
- `-N` - number of concurrent downloads (keep this low to avoid bans)
- `-f best` - requests the highest-quality combined video+audio format
- `--write-info-json` - writes per-video metadata sidecar files


In [ ]:
# =============================
# 1. SCRAPING CONFIGS
# =============================

SLEEP_REQUESTS = 3
MAX_RETRIES = 10
FRAGMENT_RETRIES = 10
CONCURRENT_DOWNLOADS = 2
METADATA_CSV_NAME = "video_metadata.csv"

output_path = Path(OUTPUT_PATH)
output_path.mkdir(parents=True, exist_ok=True)

METADATA_CSV_PATH = output_path / METADATA_CSV_NAME

# =============================
# 2. BUILD yt-dlp COMMAND
# =============================
def build_command():
    return [
        "yt-dlp",
        *video_urls,
        "--download-archive", ARCHIVE_FILE,
        "--add-header", "Referer: https://www.tiktok.com/",
        "--sleep-requests", str(SLEEP_REQUESTS),
        "--retries", str(MAX_RETRIES),
        "--fragment-retries", str(FRAGMENT_RETRIES),
        "-N", str(CONCURRENT_DOWNLOADS),
        "-o", OUTPUT_TEMPLATE,
        "-f", "best",
        "--write-info-json"
    ]

# =============================
# 3. RUN DOWNLOAD
# =============================
def run_downloader():
    cmd = build_command()
    print("Running yt-dlp...")
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f"yt-dlp failed with exit code {result.returncode}")

run_downloader()


### Save Video Metadata

After downloading, read those JSON files and save a single CSV (`video_metadata.csv`) with key TikTok metadata fields (author, post date/time, and engagement counts).


In [ ]:
def save_metadata_csv():
    records = []
    for fp in sorted(output_path.glob("*.info.json")):
        with fp.open("r", encoding="utf-8") as f:
            info = json.load(f)

        records.append({
            "meta_id": info.get("id"),
            "video_url": info.get("webpage_url") or info.get("original_url"),
            "author": info.get("uploader") or info.get("channel") or info.get("uploader_id"),
            "author_username": info.get("uploader_id"),
            "post_date": info.get("upload_date"),
            "timestamp": info.get("timestamp"),
            "view_count": info.get("view_count"),
            "like_count": info.get("like_count"),
            "comment_count": info.get("comment_count"),
            "share_count": info.get("share_count"),
            "duration_seconds": info.get("duration"),
            "description": info.get("description"),
        })

    metadata_df = pd.DataFrame(records).drop_duplicates(subset=["meta_id"], keep="first")
    metadata_df.to_csv(METADATA_CSV_PATH, index=False)
    print(f"Saved {len(metadata_df)} rows -> {METADATA_CSV_PATH}")
    return metadata_df

metadata_df = save_metadata_csv()
metadata_df.head()
